In [9]:
# Create order history for cohort and retention analysis
import pandas as pd
orders = pd.DataFrame({
    "customer_id": ["C01", "C01", "C02", "C02", "C03", "C03", "C03"],
    "order_date": [
        "2026-01-05", "2026-02-10",
        "2026-01-15", "2026-03-02",
        "2026-02-20", "2026-03-01", "2026-04-04",
    ],
    "amount": [100, 80, 50, 70, 120, 90, 100],
})
orders

,customer_id,order_date,amount
0,C01,2026-01-05,100
1,C01,2026-02-10,80
2,C02,2026-01-15,50
3,C02,2026-03-02,70
4,C03,2026-02-20,120
5,C03,2026-03-01,90
6,C03,2026-04-04,100


In [11]:
# Convert dates to monthly periods for consistent cohort grouping
orders["order_date"]=pd.to_datetime(orders["order_date"])
orders["order_month"]=orders["order_date"].dt.to_period("M")
orders

,customer_id,order_date,amount,order_month
0,C01,2026-01-05,100,2026-01
1,C01,2026-02-10,80,2026-02
2,C02,2026-01-15,50,2026-01
3,C02,2026-03-02,70,2026-03
4,C03,2026-02-20,120,2026-02
5,C03,2026-03-01,90,2026-03
6,C03,2026-04-04,100,2026-04


In [12]:
# Assign each customer to the month of their first activity
first_active=(
    orders.groupby("customer_id")["order_month"]
    .min()
    .reset_index()
    .rename(columns={"order_month":"first_active_month"})
)
first_active

,customer_id,first_active_month
0,C01,2026-01
1,C02,2026-01
2,C03,2026-02


In [13]:
# Calculate months elapsed since each customer first became active
orders=orders.merge(first_active,on="customer_id",how="left")
orders["month_offset"]=(
    (orders["order_month"].dt.year-orders["first_active_month"].dt.year)*12
    + 
    (orders["order_month"].dt.month-orders["first_active_month"].dt.month)
)
orders

,customer_id,order_date,amount,order_month,first_active_month,month_offset
0,C01,2026-01-05,100,2026-01,2026-01,0
1,C01,2026-02-10,80,2026-02,2026-01,1
2,C02,2026-01-15,50,2026-01,2026-01,0
3,C02,2026-03-02,70,2026-03,2026-01,2
4,C03,2026-02-20,120,2026-02,2026-02,0
5,C03,2026-03-01,90,2026-03,2026-02,1
6,C03,2026-04-04,100,2026-04,2026-02,2


In [19]:
# Count distinct active customers by cohort and months since first order
cohort_table=orders.pivot_table(
    index="first_active_month",
    columns="month_offset",
    values="customer_id",
    aggfunc="nunique",
    fill_value=0
)
cohort_table.index.name = "Cohort Month"
cohort_table.columns.name = "Months Since First Order"
cohort_table

Months Since First Order,0,1,2
Cohort Month,,,
2026-01,2,1,1
2026-02,1,1,1


In [20]:
# Divide active customers by cohort size to calculate retention percentages
cohort_sizes=cohort_table.iloc[:,0]
retention_rates=cohort_table.div(cohort_sizes,axis=0)*100
retention_rates

Months Since First Order,0,1,2
Cohort Month,,,
2026-01,100.0,50.0,50.0
2026-02,100.0,100.0,100.0


# 1. Why must customer counts use `nunique()` rather than row counts?

Customer counts must use `nunique()` because a single customer can place multiple orders within the same month. Row-counting methods such as `count()` or `size()` count every transaction record, which may overestimate the number of active customers.

The `nunique()` function counts only distinct customer IDs, ensuring that each customer is counted once per month.

**Example:**

| customer_id | order_month |
|---|---|
| C01 | 2026-01 |
| C01 | 2026-01 |
| C02 | 2026-01 |
| C03 | 2026-01 |

- Total transaction rows = 4
- Unique customers = 3

Therefore, `nunique()` is more appropriate for cohort analysis because it measures the number of active customers rather than the number of transactions.

# 2. Identify possible data issues such as duplicate transactions.

Several data-quality issues can affect the accuracy of cohort analysis.

| Data issue | Possible impact | Solution |
|---|---|---|
| Duplicate transactions | May inflate order counts | Identify and remove duplicate records |
| Missing customer IDs | Cannot identify customers correctly | Remove or investigate missing IDs |
| Invalid dates | Incorrect cohort assignment | Validate and convert dates |
| Incorrect customer IDs | One customer may be counted as multiple customers | Standardize customer identifiers |
| Refunded or cancelled orders | May incorrectly count invalid purchases | Filter based on order status |
| Missing transaction records | May underestimate customer activity | Investigate incomplete data |
| Timezone inconsistencies | Orders may be assigned to the wrong month | Standardize timestamps and timezones |

**Conclusion:** Data cleaning and validation are essential before performing cohort analysis to ensure accurate customer counts and reliable retention rates.

### Important notes

- Cohort retention measures distinct active customers, so `nunique()` is required when a customer can place multiple orders in one period.
- Month offsets should start at `0` for the first active month; later offsets represent months since cohort entry.
- The first-period cohort size is the denominator for each retention row, so missing later periods should be distinguished from zero activity.
- Filter cancelled, refunded, or otherwise invalid orders according to the business definition of “active” before assigning cohorts.
- Validate duplicate transactions, missing customer IDs, invalid dates, and timezone consistency before interpreting retention rates.
- Small cohorts can produce volatile percentages; report cohort sizes alongside retention rates.
